# Backtest

Loads all `WalletSelectionStrategy` objects from the workspace registry,
scores test signals for each strategy's wallet set, applies each strategy's
trigger function, and runs a full strategy sweep.  Results are compared in a
summary table and cumulative PnL plot.

**No strategy logic is defined here** — all trigger rules and parameters
live in the persisted strategy registry (written by `stage1_wallet_strategy_selection`).


In [66]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [67]:
PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]
FILL_MAX_REL_PRICE_DIFF_BY_BUCKET = {
    "0.00-0.10": 2.50,
    "0.10-0.25": 1.20,
    "0.25-0.40": 0.70,
    "0.40-0.60": 0.35,
    "0.60-0.75": 0.22,
    "0.75-0.90": 0.12,
    "0.90-1.00": 0.06,
}

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

EXEC_TAPE_TEST_PATH    = WORKSPACE_DIR / "execution_tape_test_v2.parquet"
EXEC_TAPE_TRAIN_B_PATH = WORKSPACE_DIR / "execution_tape_train_b_v2.parquet"

BACKTEST_KWARGS = dict(
    base_stake_usdc=100.0,
    latency_seconds=0,
    fill_horizon_seconds=600,
    slippage_bps=50.0,
    fee_bps=10.0,
    max_signals_per_day=20,
    dedupe_by_market=True,
    starting_capital=10_000.0,
    max_rel_price_diff_by_bucket=FILL_MAX_REL_PRICE_DIFF_BY_BUCKET,
)


## Derive train/test dates

In [68]:
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
TRAIN_A_END_DATE = TRAIN_START_DATE + (END_DATE_TRAIN - TRAIN_START_DATE) // 2
del _sample
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-02-15  TRAIN_A_END_DATE=2025-08-18


## Load strategy registry

In [69]:
from polymarket_analysis.strategy.registry import load_all_strategies
from polymarket_analysis.strategy.triggers import get_trigger

registry = load_all_strategies(WORKSPACE_DIR)
if not registry:
    raise RuntimeError("No strategies found in workspace. Run stage1_wallet_strategy_selection first.")

summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        "strategy_id": s.strategy_id,
        "name": s.name,
        "selection_method": s.selection_method,
        "num_wallets": len(s.wallets),
        "trigger_fn": s.trigger.fn_ref.split(".")[-1],
        "threshold": s.trigger.params.get("threshold"),
        "dynamic_sizing": s.trigger.params.get("dynamic_sizing"),
    })

print(f"Loaded {len(registry)} strategies:")
pd.DataFrame(summary_rows)


Loaded 8 strategies:


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,early_entry__all_open_buys,early_entry | all open-buys (fixed stake),cohort_early_entry,47,all_open_buys,NaN,False
1,early_entry__score_threshold,early_entry | score >= 0.85 (Kelly),cohort_early_entry,47,score_threshold,0.85,True
2,quality_core__all_open_buys,quality_core | all open-buys (fixed stake),skill_sweep,50,all_open_buys,NaN,False
3,quality_core__score_threshold,quality_core | score >= 0.85 (Kelly),skill_sweep,50,score_threshold,0.85,True
4,smooth_pnl__all_open_buys,smooth_pnl | all open-buys (fixed stake),cohort_smooth_pnl,50,all_open_buys,NaN,False
5,smooth_pnl__score_threshold,smooth_pnl | score >= 0.85 (Kelly),cohort_smooth_pnl,50,score_threshold,0.85,True
6,volatility__all_open_buys,volatility | all open-buys (fixed stake),volatility,100,all_open_buys,NaN,False
7,volatility__score_threshold,volatility | score >= 0.85 (Kelly),volatility,100,score_threshold,0.85,True


## Load calibration signals

In [70]:
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH         = WORKSPACE_DIR / "signal_events_test.parquet"

train_b_open_buys = pd.read_parquet(CALIBRATION_SIGNALS_PATH)
test_open_buys    = pd.read_parquet(TEST_SIGNALS_PATH)
print(f"train_b signals: {len(train_b_open_buys):,}  test signals: {len(test_open_buys):,}")


train_b signals: 823  test signals: 1,061


## Rebuild calibration tables

In [71]:
from polymarket_analysis.signal.scorer import (
    build_calibration_tables,
    apply_signal_score,
    select_signal_threshold,
)

price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
print(f"Global edge: {global_edge:.4f}  Signal threshold: {SIGNAL_THRESHOLD:.2f}")


Global edge: 0.1409  Signal threshold: 0.85


## Build per-strategy scored signals

For each strategy, build wallet profiles and signal events **from that strategy's wallet set**,
then score them. This ensures each strategy is evaluated on its own wallet cohort.

In [72]:
from polymarket_analysis.signal.builder import build_wallet_profiles, build_signal_events

strategy_test_signals:      dict[str, pd.DataFrame] = {}
strategy_train_ref_signals: dict[str, pd.DataFrame] = {}

# Group strategies by wallet set identity to avoid redundant profile/signal builds.
# Two strategies share signals when their wallet DataFrames are identical.
_signals_cache: dict[frozenset, tuple[pd.DataFrame, pd.DataFrame]] = {}

for sid, strategy in registry.items():
    wallet_key = frozenset(strategy.wallets["wallet"])
    if wallet_key not in _signals_cache:
        profiles = build_wallet_profiles(
            dataset, strategy.wallets, period="full_train",
            end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
        )
        _, c_test = build_signal_events(
            dataset, profiles, period="test",
            end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
            price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
            tx_hash_column=TRIGGER_TX_HASH_COLUMN,
        )
        _, c_train = build_signal_events(
            dataset, profiles, period="train_b",
            end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
            price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
            tx_hash_column=TRIGGER_TX_HASH_COLUMN,
        )
        scored_test  = apply_signal_score(c_test,  price_table, consensus_table)
        scored_train = apply_signal_score(c_train, price_table, consensus_table)
        _signals_cache[wallet_key] = (scored_test, scored_train)
    strategy_test_signals[sid], strategy_train_ref_signals[sid] = _signals_cache[wallet_key]

{sid: len(df) for sid, df in strategy_test_signals.items()}


{'early_entry__all_open_buys': 5355,
 'early_entry__score_threshold': 5355,
 'quality_core__all_open_buys': 5638,
 'quality_core__score_threshold': 5638,
 'smooth_pnl__all_open_buys': 39883,
 'smooth_pnl__score_threshold': 39883,
 'volatility__all_open_buys': 129142,
 'volatility__score_threshold': 129142}

## Build / load execution tapes

In [73]:
from polymarket_analysis.backtest.execution_tape import (
    build_execution_tape,
    build_tape_groups,
    normalize_execution_tape,
)

def _collect_cids(signals_dict):
    parts = [df["condition_id"] for df in signals_dict.values() if not df.empty]
    if not parts:
        return []
    return list(pd.concat(parts, ignore_index=True).drop_duplicates())

all_test_cids  = _collect_cids(strategy_test_signals)
all_train_cids = _collect_cids(strategy_train_ref_signals)

if EXEC_TAPE_TEST_PATH.exists():
    execution_tape_test = pd.read_parquet(EXEC_TAPE_TEST_PATH)
else:
    execution_tape_test = build_execution_tape(
        dataset, all_test_cids, tx_hash_column=TRIGGER_TX_HASH_COLUMN
    )
    execution_tape_test.to_parquet(EXEC_TAPE_TEST_PATH, index=False)

if EXEC_TAPE_TRAIN_B_PATH.exists():
    execution_tape_train_b = pd.read_parquet(EXEC_TAPE_TRAIN_B_PATH)
else:
    execution_tape_train_b = build_execution_tape(
        dataset, all_train_cids,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
        start_date=TRAIN_A_END_DATE + datetime.timedelta(days=1),
        end_date=END_DATE_TRAIN,
    )
    execution_tape_train_b.to_parquet(EXEC_TAPE_TRAIN_B_PATH, index=False)

execution_tape_test["tape_dt"]    = pd.to_datetime(execution_tape_test["tape_dt"],    utc=True)
execution_tape_train_b["tape_dt"] = pd.to_datetime(execution_tape_train_b["tape_dt"], utc=True)
tape_groups           = build_tape_groups(normalize_execution_tape(execution_tape_test))
tape_groups_train_ref = build_tape_groups(normalize_execution_tape(execution_tape_train_b))
print(f"tape markets – test: {len(tape_groups):,}  train_b: {len(tape_groups_train_ref):,}")


tape markets – test: 2,018  train_b: 3,241


## Full strategy sweep

For each strategy:
1. Resolve the trigger function from its `fn_ref`.
2. Apply it to the scored signals to get a boolean mask.
3. Run `backtest_strategy` with `dynamic_sizing` from the strategy's params.


In [74]:
from polymarket_analysis.backtest.strategy import (
    backtest_strategy,
    summarize_backtest,
    build_trigger_ledger,
)

def _run_strategy_registry(registry, signals_dict, tapes):
    """Run backtest for every strategy in the registry.

    Parameters
    ----------
    registry    : {strategy_id -> WalletSelectionStrategy}
    signals_dict: {strategy_id -> scored_signals DataFrame}
    tapes       : tape_groups dict

    Returns
    -------
    runs    : {strategy_id -> {trades, daily, unfilled_triggers, theoretical_daily, spec}}
    summary : DataFrame
    """
    runs, rows = {}, []
    for sid, strategy in registry.items():
        scored_df = signals_dict.get(sid, pd.DataFrame())
        if scored_df.empty:
            continue

        # Resolve and apply trigger function
        trigger_fn = get_trigger(strategy.trigger.fn_ref)
        mask = trigger_fn(scored_df, strategy.trigger.params)

        dynamic_sizing = bool(strategy.trigger.params.get("dynamic_sizing", True))

        trades_bt, daily_bt, unfilled_bt, theoretical_bt = backtest_strategy(
            scored_df, mask, tapes,
            strategy.strategy_id,
            strategy.trigger.fn_ref.split(".")[-1],
            dynamic_sizing=dynamic_sizing,
            cohort_name=strategy.metadata.get("cohort", strategy.selection_method),
            **BACKTEST_KWARGS,
        )
        runs[sid] = {
            "trades": trades_bt,
            "daily": daily_bt,
            "unfilled_triggers": unfilled_bt,
            "theoretical_daily": theoretical_bt,
            "strategy": strategy,
        }
        row = summarize_backtest(trades_bt, daily_bt).to_dict()
        n_triggered = len(trades_bt) + len(unfilled_bt)
        row.update({
            "strategy_id": sid,
            "name": strategy.name,
            "selection_method": strategy.selection_method,
            "trigger_fn": strategy.trigger.fn_ref.split(".")[-1],
            "dynamic_sizing": dynamic_sizing,
            "triggered_signals": n_triggered,
            "unfilled_triggers": len(unfilled_bt),
            "fill_rate": len(trades_bt) / n_triggered if n_triggered else float("nan"),
        })
        rows.append(row)

    if not rows:
        return runs, pd.DataFrame()
    summary = (
        pd.DataFrame(rows)
        .set_index("strategy_id")
        .sort_values("net_roi_on_stake", ascending=False)
    )
    return runs, summary


strategy_runs,           strategy_summary           = _run_strategy_registry(
    registry, strategy_test_signals,      tape_groups
)
strategy_runs_train_ref, strategy_summary_train_ref = _run_strategy_registry(
    registry, strategy_train_ref_signals, tape_groups_train_ref
)

trigger_ledgers = {
    n: build_trigger_ledger(r["trades"], r["unfilled_triggers"])
    for n, r in strategy_runs.items()
}


## Summary table

In [75]:
pd.set_option("display.max_columns", None)
strategy_summary


,filled_trades,net_pnl_usdc,net_roi_on_stake,win_rate,name,selection_method,trigger_fn,dynamic_sizing,triggered_signals,unfilled_triggers,fill_rate
strategy_id,,,,,,,,,,,
early_entry__all_open_buys,0.0,0.0,NaN,NaN,early_entry | all open-buys (fixed stake),cohort_early_entry,all_open_buys,False,471,471,0.0
early_entry__score_threshold,0.0,0.0,NaN,NaN,early_entry | score >= 0.85 (Kelly),cohort_early_entry,score_threshold,True,54,54,0.0
quality_core__all_open_buys,0.0,0.0,NaN,NaN,quality_core | all open-buys (fixed stake),skill_sweep,all_open_buys,False,471,471,0.0
quality_core__score_threshold,0.0,0.0,NaN,NaN,quality_core | score >= 0.85 (Kelly),skill_sweep,score_threshold,True,204,204,0.0
smooth_pnl__all_open_buys,0.0,0.0,NaN,NaN,smooth_pnl | all open-buys (fixed stake),cohort_smooth_pnl,all_open_buys,False,480,480,0.0
smooth_pnl__score_threshold,0.0,0.0,NaN,NaN,smooth_pnl | score >= 0.85 (Kelly),cohort_smooth_pnl,score_threshold,True,223,223,0.0
volatility__all_open_buys,0.0,0.0,NaN,NaN,volatility | all open-buys (fixed stake),volatility,all_open_buys,False,480,480,0.0
volatility__score_threshold,0.0,0.0,NaN,NaN,volatility | score >= 0.85 (Kelly),volatility,score_threshold,True,460,460,0.0


## Train-ref summary (in-sample calibration check)

In [76]:
strategy_summary_train_ref


,filled_trades,net_pnl_usdc,net_roi_on_stake,win_rate,avg_signal_score,avg_kelly_fraction,avg_fill_delay_minutes,opposite_fill_share,max_drawdown_usdc,name,selection_method,trigger_fn,dynamic_sizing,triggered_signals,unfilled_triggers,fill_rate
strategy_id,,,,,,,,,,,,,,,,
smooth_pnl__all_open_buys,14.0,243.548736,0.173963,0.571429,0.543603,NaN,3.390476,0.071429,147.219250,smooth_pnl | all open-buys (fixed stake),cohort_smooth_pnl,all_open_buys,False,1024,1010,0.013672
volatility__score_threshold,5.0,650.647060,0.173506,0.400000,0.875471,0.1,2.300000,0.278515,0.000000,volatility | score >= 0.85 (Kelly),volatility,score_threshold,True,938,933,0.005330
quality_core__all_open_buys,8.0,96.590381,0.120738,0.500000,0.569800,NaN,3.145833,0.220907,41.658454,quality_core | all open-buys (fixed stake),skill_sweep,all_open_buys,False,1036,1028,0.007722
smooth_pnl__score_threshold,7.0,-1359.630600,-0.258977,0.142857,0.875007,0.1,2.352381,0.097856,1501.500000,smooth_pnl | score >= 0.85 (Kelly),cohort_smooth_pnl,score_threshold,True,592,585,0.011824
volatility__all_open_buys,8.0,-211.439112,-0.264299,0.375000,0.595563,NaN,3.658333,0.278260,300.300000,volatility | all open-buys (fixed stake),volatility,all_open_buys,False,1853,1845,0.004317
early_entry__all_open_buys,7.0,-213.672769,-0.305247,0.285714,0.562989,NaN,2.142857,0.252465,141.758454,early_entry | all open-buys (fixed stake),cohort_early_entry,all_open_buys,False,1031,1024,0.006790
early_entry__score_threshold,1.0,-750.750000,-1.001000,0.000000,0.880974,0.1,4.233333,0.000000,0.000000,early_entry | score >= 0.85 (Kelly),cohort_early_entry,score_threshold,True,84,83,0.011905
quality_core__score_threshold,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,quality_core | score >= 0.85 (Kelly),skill_sweep,score_threshold,True,607,607,0.000000


## Cumulative PnL comparison

In [77]:
from polymarket_analysis.visualization.backtest_plots import (
    plot_strategy_test,
    plot_strategy_train,
)

fig_test = plot_strategy_test(
    strategy_runs,
    title="Strategy registry — test period (1h resolution)",
)
fig_test.show(renderer="browser")

fig_train = plot_strategy_train(
    strategy_runs_train_ref,
    title="Strategy registry — train-B reference period (1h resolution)",
)
fig_train.show(renderer="browser")

## Placebo benchmark

Random-wallet baseline to verify the selected strategies beat random chance.

In [78]:
METRICS_FULL_PATH    = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
SELECTED_WALLETS_PATH = WORKSPACE_DIR / "selected_wallets_v2.parquet"

full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
selected_wallets   = pd.read_parquet(SELECTED_WALLETS_PATH)

rng = np.random.default_rng(42)
eligible_placebo = full_train_metrics[
    (full_train_metrics["open_buy_trades"] >= 20)
    & (full_train_metrics["volume"] >= 500.0)
    & (full_train_metrics.get("distinct_markets",
       pd.Series(0, index=full_train_metrics.index)) >= 8)
    & (full_train_metrics.get("recent_open_buy_trades",
       pd.Series(0, index=full_train_metrics.index)) >= 3)
].copy()

# Compare against the best skill-sweep score-threshold strategy
SCORE_STRATEGY_ID = next(
    (sid for sid in strategy_runs if "quality_core__score_threshold" in sid), None
)

if SCORE_STRATEGY_ID and len(eligible_placebo) >= len(selected_wallets):
    placebo_wallets = eligible_placebo.sample(
        n=len(selected_wallets), random_state=42
    ).copy().reset_index(drop=True)
    placebo_wallets["wallet_quality"] = rng.random(len(placebo_wallets))
    placebo_profiles = build_wallet_profiles(
        dataset, placebo_wallets, period="full_train",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
    )
    _, placebo_test_open_buys = build_signal_events(
        dataset, placebo_profiles, period="test",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    placebo_scored = apply_signal_score(placebo_test_open_buys, price_table, consensus_table)
    if not placebo_scored.empty and "signal_score" in placebo_scored.columns:
        placebo_trades_bt, placebo_daily_bt, _, _ = backtest_strategy(
            placebo_scored,
            mask=placebo_scored["signal_score"] >= SIGNAL_THRESHOLD,
            tape_groups=tape_groups,
            strategy_name="placebo_random_wallets",
            trigger_rule=f"placebo signal_score >= {SIGNAL_THRESHOLD:.2f}",
            dynamic_sizing=True,
            cohort_name="placebo",
            **BACKTEST_KWARGS,
        )
        test_trades_bt = strategy_runs[SCORE_STRATEGY_ID]["trades"]
        test_daily_bt  = strategy_runs[SCORE_STRATEGY_ID]["daily"]
        placebo_compare = pd.DataFrame([
            summarize_backtest(test_trades_bt,     test_daily_bt    ).rename(SCORE_STRATEGY_ID),
            summarize_backtest(placebo_trades_bt, placebo_daily_bt ).rename("placebo_random_wallets"),
        ])
        placebo_compare
    else:
        print("Placebo signals empty after scoring")
else:
    print("Not enough eligible wallets for placebo or no score-threshold strategy found")
